# Preliminary verification of the proposed method on track 1

Assignment 2, section 9. The proposed method is reversal-equivariant LoRA on a decoder LLM (`dico_nli.method`): a frozen Qwen2.5 backbone scored on four option letters, LoRA adapters trained with a loss that ties the prediction for `(a, b)` to the one for `(b, a)`, and symmetrized decoding at prediction time.

This notebook is a **pipeline check, not a performance claim**. Every trained system here is one seed. It shows that data loads, prompts are built, the model trains, predictions come out in the official format and the official scorer accepts them.

**What this notebook does**

1. Loads track 1 and reports the split sizes.
2. Shows the prompt and checks its length against the limit.
3. Scores dev zero-shot with the 0.5B model on this machine's CPU (cached under `artifacts/` after the first run).
4. Reads the 1.5B runs fetched from the Modal volume, when they are there: zero-shot, LoRA with lambda 0, LoRA with lambda 1.
5. Scores every system with the official scorer, once per decoding (single pass and symmetrized).
6. Records predicted-label counts, pair outcomes, the training history, ten example predictions, and every warning.

**Protocol, fixed in the plan (issue 13) before the first run**

| Setting | Value |
|---|---|
| Backbone | `Qwen/Qwen2.5-1.5B-Instruct`, frozen; `Qwen/Qwen2.5-0.5B-Instruct` for the CPU check |
| Trainable | LoRA on `q_proj`, `k_proj`, `v_proj`, `o_proj`: rank 16, alpha 32, dropout 0.05 |
| Input | a fixed chat prompt with the two phrases and four lettered relations; limit 160 tokens, never truncated |
| Output | softmax over the next-token logits of the letters A, B, C, D |
| Loss | cross-entropy on both orders plus lambda times the Jensen-Shannon divergence between the two predictions |
| Optimisation | AdamW, learning rate 2e-4, weight decay 0.01, 16 pairs per step, 3 epochs, linear schedule with 6 % warm-up, gradient clipping at 1.0 |
| Epoch selection | best weighted F1 (single pass) on a 20 % slice of train, split by source pair with the run's seed; dev is never seen during fitting |
| Seed | 13 |
| Compute | Modal, launched by `scripts/modal/train_decoder.py`; GPU, precision and timing are read from each run's `run.json` |

**Where the results go**

| Output | Path |
|---|---|
| Scores, label counts, pair outcomes, history, examples | `experiments/2026-11-14-decoder-preliminary-track1/tables/` |
| Predictions and scorer reports per system and decoding | `predictions/`, `scores/<system>__<decoding>/` in that folder |
| Per-system run records | `runs/` in that folder |
| Checks and warnings | `checks.json` in that folder |

## 1. Setup

Imports and the run parameters. `MODAL_RUN_ID` is the folder name used on the Modal volume and under `artifacts/`.

In [ ]:
import json
import platform
import random
import shutil
import time
from dataclasses import asdict
from datetime import date

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from dico_nli.analysis import PAIR_OUTCOMES, analyze_predictions
from dico_nli.data import DATA_ROOT, LABELS, REPO_ROOT, grouped_split, load_reference, reversed_text_twins
from dico_nli.experiments import experiment_dir, write_config, write_results_table
from dico_nli.method.decoder import LETTERS, DecoderClassifier, DecoderConfig, render_prompt
from dico_nli.method.reversal import symmetrize
from dico_nli.scoring import score, write_predictions

TRACK = 1
SEED = 13
RUN_DATE = date(2026, 11, 14)
OVERWRITE = True
LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LOCAL_SYSTEM = "qwen2.5-0.5b-zero-shot"
MODAL_RUN_ID = "2026-11-14-decoder-preliminary-track1"
MODAL_SYSTEMS = ["qwen2.5-1.5b-zero-shot", "qwen2.5-1.5b-lora-lambda0-seed13", "qwen2.5-1.5b-lora-lambda1-seed13"]
ARTIFACTS = REPO_ROOT / "artifacts" / "decoder-preliminary-track1"
LOCAL_CACHE = ARTIFACTS / "local" / LOCAL_SYSTEM
FETCHED = ARTIFACTS / MODAL_RUN_ID
COLUMNS = list(LABELS)
warnings = []

## 2. Data and splits

The official train and dev files. The validation slice is 20 % of train, split by source pair so both directions of a pair stay on the same side. Dev is the course test set and is only ever predicted on.

In [ ]:
train = load_reference(TRACK, "train")
dev = load_reference(TRACK, "dev")
fit_part, holdout = grouped_split(train, 0.2, SEED)
gold = DATA_ROOT / "dev" / f"dico_nli_dev_track{TRACK}_reference.csv"
out = experiment_dir("decoder-preliminary-track1", day=RUN_DATE, overwrite=OVERWRITE)
tables, runs = out / "tables", out / "runs"
for folder in (tables, runs):
    folder.mkdir(exist_ok=True)

splits = {"train (all)": train, "train, fit part": fit_part, "train, validation part": holdout, "dev": dev}
split_sizes = pd.DataFrame(
    [{"split": name, "instances": len(rows), "pairs": len({i.pair_id for i in rows}),
      **{label: sum(i.label == label for i in rows) for label in LABELS}} for name, rows in splits.items()]
)
assert not {i.pair_id for i in fit_part} & {i.pair_id for i in holdout}, "a pair sits on both sides of the split"
split_sizes.to_csv(tables / "split_sizes.csv", index=False)
split_sizes

The method trains on both orders of every pair and treats a reversed NEGATIVE_OTHER pair as NEGATIVE_OTHER. That is an assumption: the organizers define no reverse for that label. The table lists every negative whose two phrases also occur in the opposite order in the same file, with the label of that reversed copy. It shows how little the data can say about the assumption.

In [ ]:
negative_rows = []
for name, rows in (("train", train), ("dev", dev)):
    negatives = [i for i in rows if i.label == "NEGATIVE_OTHER"]
    twins = reversed_text_twins(negatives, rows)
    negative_rows.append({"split": name, "negatives": len(negatives),
                          "with_reversed_copy": len({a.instance_id for a, _ in twins}),
                          "reversed_copies": "; ".join(f"{b.text1} / {b.text2}: {b.label}" for _, b in twins)})
negatives_reversed = pd.DataFrame(negative_rows)
negatives_reversed.to_csv(tables / "negatives_with_reversed_copy.csv", index=False)
negatives_reversed

## 3. The prompt

The exact text the model reads for one training pair, the token id of each option letter, and the longest prompt over both orders of every train and dev instance. A prompt over the limit is an error, never a truncation, because truncation would cut off the answer position.

In [ ]:
local = DecoderClassifier(DecoderConfig(model_name=LOCAL_MODEL, seed=SEED))
example = render_prompt(local.tokenizer, train[0].text1, train[0].text2)
(out / "prompt_example.txt").write_text(example + "\n", encoding="utf-8")
print(example)

longest = local.max_prompt_tokens([*train, *dev])
prompt_facts = {
    "letters": dict(zip(LETTERS, LABELS)),
    "letter_token_ids": dict(zip(LETTERS, local.letter_ids)),
    "longest_prompt_tokens": longest,
    "limit_tokens": local.config.max_length,
    "padding_side": local.tokenizer.padding_side,
}
assert longest <= local.config.max_length
prompt_facts

## 4. Zero-shot on this machine: the 0.5B model on CPU

The smallest end-to-end proof. No training: the frozen 0.5B model scores every dev instance in both orders. The probabilities are cached under `artifacts/` because the pass takes several minutes on a CPU; delete that folder to compute them again.

In [ ]:
def frames(folder):
    return (pd.read_csv(folder / "probabilities_forward.csv"), pd.read_csv(folder / "probabilities_reverse.csv"),
            json.loads((folder / "run.json").read_text(encoding="utf-8")))


if not (LOCAL_CACHE / "run.json").exists():
    started = time.time()
    forward, reverse = local.predict_both_orders(dev)
    seconds = time.time() - started
    LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
    forward.to_csv(LOCAL_CACHE / "probabilities_forward.csv", index=False)
    reverse.to_csv(LOCAL_CACHE / "probabilities_reverse.csv", index=False)
    record = {
        "system": LOCAL_SYSTEM, "mode": "zero-shot", "config": asdict(local.config), "dev_size": len(dev),
        "predict_seconds": round(seconds, 1), "device": "cpu", "cpu": platform.processor(),
        "threads": torch.get_num_threads(), "computed_on": date.today().isoformat(),
        "total_parameters": sum(p.numel() for p in local.model.parameters()),
        "precision": str(next(local.model.parameters()).dtype), "versions": {"torch": torch.__version__},
    }
    (LOCAL_CACHE / "run.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")

systems = {LOCAL_SYSTEM: frames(LOCAL_CACHE)}
{key: systems[LOCAL_SYSTEM][2][key] for key in ("predict_seconds", "device", "threads", "total_parameters", "precision", "computed_on")}

## 5. The 1.5B runs from Modal

Zero-shot, LoRA with lambda 0 and LoRA with lambda 1, seed 13, fetched with `modal volume get`. A system whose files are not there is skipped and listed under the warnings; nothing is filled in for it.

In [ ]:
for name in MODAL_SYSTEMS:
    folder = FETCHED / name
    missing = [f for f in ("probabilities_forward.csv", "probabilities_reverse.csv", "run.json") if not (folder / f).exists()]
    if missing:
        warnings.append(f"{name}: not scored, {missing} missing under artifacts/decoder-preliminary-track1/{MODAL_RUN_ID}/{name}")
        continue
    systems[name] = frames(folder)

for name, (_, _, record) in systems.items():
    (runs / f"{name}.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print("scored here:", list(systems))
warnings

## 6. Official scores, single pass and symmetrized

Each system is scored twice. **Single pass** takes the argmax of `p(. | a, b)`. **Symmetrized** averages it with the reversed order mapped back, `1/2 [p(y | a, b) + p(Rev(y) | b, a)]`, so its SoftCons is 1.0 by construction; the evidence for that decoding is weighted F1 and HardCons.

In [ ]:
dev_ids = [i.instance_id for i in dev]
rows, score_rows, predictions = {}, [], {}
for name, (forward, reverse, record) in systems.items():
    assert list(forward["instance_id"]) == dev_ids and list(reverse["instance_id"]) == dev_ids, name
    p_forward, p_reverse = forward[COLUMNS].to_numpy(), reverse[COLUMNS].to_numpy()
    assert np.allclose(p_forward.sum(axis=1), 1.0, atol=1e-4) and np.allclose(p_reverse.sum(axis=1), 1.0, atol=1e-4), name
    for decoding, probabilities in (("single", p_forward), ("symmetrized", symmetrize(p_forward, p_reverse))):
        key = f"{name}__{decoding}"
        predictions[key] = {i: LABELS[k] for i, k in zip(dev_ids, probabilities.argmax(axis=1))}
        result = score(gold, write_predictions(out / "predictions" / f"{key}.csv", predictions[key]), out / "scores" / key)
        report = json.loads(result.report_path.read_text(encoding="utf-8"))
        rows[f"{name}, {decoding}"] = result
        score_rows.append({
            "system": name, "decoding": decoding,
            "weighted_f1": result.weighted_f1, "soft_cons": result.soft_cons, "hard_cons": result.hard_cons,
            "macro_f1": report["classification"]["macro_f1"], "accuracy": report["classification"]["accuracy"],
            **{f"f1_{label}": report["classification"]["per_label"][label]["f1"] for label in LABELS},
            "reversible_pairs": report["consistency"]["reversible_pairs"],
            "soft_consistent_pairs": report["consistency"]["soft_consistent_pairs"],
            "hard_correct_pairs": report["consistency"]["hard_correct_pairs"],
        })
scores = pd.DataFrame(score_rows)
scores.to_csv(tables / "dev_scores.csv", index=False)
scores.round(3)

For context, the Assignment 1 systems on the same dev file, read from their own recorded runs (the encoder row is the mean of three seeds).

In [ ]:
def official(path):
    report = json.loads(path.read_text(encoding="utf-8"))
    return {key: report[key] for key in ("weighted_f1", "soft_cons", "hard_cons")}


a1 = REPO_ROOT / "experiments"
encoder = pd.read_csv(a1 / "2026-09-29-deberta-base-track1" / "tables" / "dev_scores_summary.csv", index_col=0)
context = {
    "majority": official(a1 / "2026-09-29-trivial-baselines-track1" / "scores" / "majority" / "scores.json"),
    "always-EQUIVALENCE": official(a1 / "2026-09-29-trivial-baselines-track1" / "scores" / "always-equivalence" / "scores.json"),
    "both + logreg": official(a1 / "2026-09-29-ml-baseline-track1" / "scores" / "both+logreg" / "scores.json"),
    "DeBERTa-v3-base, mean of 3 seeds": {key: encoder.loc["mean", key] for key in ("weighted_f1", "soft_cons", "hard_cons")},
}
comparison = pd.concat([
    pd.DataFrame(context).T,
    scores.assign(system=scores["system"] + ", " + scores["decoding"]).set_index("system")[["weighted_f1", "soft_cons", "hard_cons"]],
])
comparison.to_csv(tables / "comparison_with_assignment_1.csv")
comparison.round(3)

## 7. What the systems predict

Predicted label counts next to the gold counts. A model that puts nearly everything on one label is self-consistent and wrong, which the scores alone can hide.

In [ ]:
gold_labels = {i.instance_id: i.label for i in dev}
counts = pd.DataFrame({"gold": pd.Series(list(gold_labels.values())).value_counts()})
for key, mapping in predictions.items():
    counts[key] = pd.Series(list(mapping.values())).value_counts()
counts = counts.reindex(COLUMNS).fillna(0).astype(int)
counts.to_csv(tables / "predicted_label_counts.csv")
counts.T

## 8. Pair outcomes

Every reversible dev pair falls into one of four outcomes (`dico_nli.analysis`): both directions correct, consistent but wrong, inconsistent with one direction correct, inconsistent with both wrong.

In [ ]:
outcome_rows = []
for key, mapping in predictions.items():
    pairs, _ = analyze_predictions(dev, mapping)
    tally = pairs["outcome"].value_counts()
    outcome_rows.append({"system": key, **{outcome: int(tally.get(outcome, 0)) for outcome in PAIR_OUTCOMES}, "pairs": len(pairs)})
pair_outcomes = pd.DataFrame(outcome_rows)
pair_outcomes.to_csv(tables / "pair_outcomes.csv", index=False)
pair_outcomes

## 9. Training record

Per-epoch history of every trained system, with the epoch that was kept, and the cost of every run: seconds, GPU, peak memory, precision, trainable parameters.

In [ ]:
history_rows, cost_rows = [], []
for name, (_, _, record) in systems.items():
    for row in record.get("history", []):
        history_rows.append({"system": name, **row, "kept": row["epoch"] == record["best_epoch"]})
    cost_rows.append({
        "system": name, "mode": record["mode"], "device": record.get("gpu", record.get("device")),
        "precision": record.get("precision"), "train_seconds": record.get("train_seconds", 0.0),
        "predict_seconds": record["predict_seconds"], "peak_gpu_memory_gb": record.get("peak_gpu_memory_gb"),
        "trainable_parameters": record.get("trainable_parameters", 0), "total_parameters": record["total_parameters"],
        "fit_size": record.get("fit_size"), "holdout_size": record.get("holdout_size"), "best_epoch": record.get("best_epoch"),
    })
history = pd.DataFrame(history_rows)
history.to_csv(tables / "training_history.csv", index=False)
costs = pd.DataFrame(cost_rows)
costs.to_csv(tables / "run_costs.csv", index=False)
if history.empty:
    warnings.append("no trained system was scored; training_history.csv is empty")
display(history.round(4))
costs

## 10. Ten example predictions

Ten dev instances drawn with the run seed, with the gold label and every system's prediction in both decodings.

In [ ]:
picked = random.Random(SEED).sample(range(len(dev)), 10)
examples = pd.DataFrame([
    {"text1": dev[k].text1, "text2": dev[k].text2, "gold": dev[k].label,
     **{key: mapping[dev[k].instance_id] for key, mapping in predictions.items()}}
    for k in picked
])
examples.to_csv(tables / "example_predictions.csv", index=False)
examples

## 11. Checks and warnings

What was verified in this run and everything that did not go as planned. A symmetrized decoding whose SoftCons is not 1.0 would be a bug in the reversal code, so it is checked for every system.

In [ ]:
symmetrized = scores[scores["decoding"] == "symmetrized"]
for _, row in symmetrized.iterrows():
    if row["soft_cons"] != 1.0:
        warnings.append(f"{row['system']}: symmetrized SoftCons is {row['soft_cons']}, expected 1.0")
checks = {
    "systems_scored": list(systems),
    "systems_expected": [LOCAL_SYSTEM, *MODAL_SYSTEMS],
    "prompt": prompt_facts,
    "every_system_predicts_every_dev_instance": all(set(mapping) == set(dev_ids) for mapping in predictions.values()),
    "symmetrized_soft_cons_is_one": bool((symmetrized["soft_cons"] == 1.0).all()),
    "pairs_on_both_sides_of_the_validation_split": 0,
    "warnings": warnings,
}
(out / "checks.json").write_text(json.dumps(checks, indent=2) + "\n", encoding="utf-8")
checks

## 12. Run record

`config.json` records the seed, the settings of every scored system, the data commit, this repository's commit and the command. `results.md` holds the official scores in the form the report cites.

In [ ]:
write_config(
    out, seed=SEED, track=TRACK,
    command="PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-2/01_preliminary_verification.ipynb",
    extra={"modal_run_id": MODAL_RUN_ID, "launcher": "scripts/modal/train_decoder.py",
           "systems": {name: record["config"] for name, (_, _, record) in systems.items()},
           "holdout_fraction": 0.2, "fit_size": len(fit_part), "holdout_size": len(holdout), "dev_size": len(dev),
           "local_model": LOCAL_MODEL},
)
print(write_results_table(out, rows, title=f"Decoder method, preliminary verification, track {TRACK} dev").read_text(encoding="utf-8"))